[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/13.%20The%20Quay-Side%20Ancillary%20Service%20Scheduling%20Problem/P13-Tier-6_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 13. The Quay-Side Ancillary Service Scheduling Problem
## Autonomous Multi-Agent Ecosystem

### Purpose and scope

This notebook simulates ancillary-service scheduling as interactions among autonomous service agents. Agents submit bids for schedule slots; a sealed-bid auction selects a provider; a simulated contract records the agreement; and execution results update reputation and agent strategy. The notebook's objective is to make this sequence explicit and inspectable, not to solve a centralized mathematical optimization model.

**Inputs** include six service-agent configurations, a 24-hour set of routine, priority, and emergency slots, resource requirements generated for each slot, and simulation rules for bidding and contract execution. **Outputs** include auction and contract records, agent and system metrics, and a performance dashboard.

### Modeling assumptions

- Each agent represents one service provider and controls its own stated resources. Slot requirements use generic categories; an explicit mapping translates those categories into each provider's service-specific resource names.
- A provider can bid only when it can supply every resource in its mapped commitment. A successful award reserves those resources until contract execution.
- The sealed-bid auction awards the slot to the highest bid. The second-highest bid is recorded as a market-clearing reference price; payment in this simulation is the winner's bid, not that reference price.
- The ledger and contracts are in-memory simulations. They are not a blockchain, do not provide cryptographic security, and do not model real payment infrastructure.
- Slot execution quality and slot resource requirements use Python's random-number generator. No fixed random seed is specified, so individual simulation runs need not produce identical results.
- Time is represented by hourly simulation steps. Service duration is one hour; more detailed overlaps, travel, labor rules, service dependencies, and real-world uncertainty are outside this model.

### How to read the model

A **state** is the information needed to describe the simulation at a point in time: agent resources and reputation, slot allocation, contracts, and the ledger. An **action** is a bid, resource reservation, contract execution, or strategy update. The simulation moves from one state to the next through these actions; it does not calculate a global optimum.

The main sequence is: initialize agents and slots; collect sealed bids; award a slot and create a contract; execute contracts when due; update resources, reputation, and metrics; then inspect the resulting time series. This agent-based process is useful for exploring decentralized coordination, but its metrics should be interpreted as outcomes of the stated rules and one stochastic run—not as proof of optimality or general system performance.

### 1. Dependencies

In [ ]:
from collections import defaultdict, deque
from datetime import datetime, timedelta
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

print("Dependencies imported successfully.")
print("The simulation uses agent-based auctions and contract execution.")

### 2. Service agents

Each `ServiceAgent` stores its own resources, preferences, reputation, and negotiation strategy. Utility is a weighted score—not a probability or monetary value. For a slot \(s\) and resource commitment \(r\), the agent computes

$$U(s,r)=w_e E(s)+w_u R(r)+w_c C(s)+w_q Q(s),$$

where \(E\) measures the time preference, \(R\) measures resource fit, \(C\) scores estimated cost, and \(Q\) represents service quality. The weights are the agent's objective weights. The code's `calculate_utility` implements this sum; `make_bid` applies the acceptance threshold and strategy-specific bid rule. A bid is rejected if any committed resource is unavailable.

In [ ]:
class ServiceAgent:
    """A provider agent with local resources, objectives, and bidding rules."""

    def __init__(self, agent_id, service_type, capabilities, resources):
        self.agent_id = agent_id
        self.service_type = service_type
        self.capabilities = capabilities
        self.resources = resources.copy()
        self.current_schedule = {}
        self.resource_availability = resources.copy()
        self.reputation_score = 1.0
        self.success_history = deque(maxlen=100)
        self.negotiation_history = deque(maxlen=50)
        self.learning_rate = 0.1
        self.exploration_rate = 0.2
        self.strategy_weights = {
            "aggressive": 0.3,
            "cooperative": 0.4,
            "conservative": 0.3,
        }
        self.objectives = {
            "schedule_efficiency": 0.4,
            "resource_utilization": 0.3,
            "cost_minimization": 0.2,
            "service_quality": 0.1,
        }
        self.negotiation_strategy = "cooperative"
        self.min_acceptable_utilization = 0.6
        self.max_bid_price = 1000

    def calculate_utility(self, schedule_slot, resource_requirements):
        """Score a slot using the agent's weighted objectives."""
        time_preference = max(
            0.0, 1.0 - abs(schedule_slot["time"] - 12) / 12
        )
        resource_fit = 1.0
        for resource_type, required in resource_requirements.items():
            available = self.resource_availability.get(resource_type, 0)
            if required == 0:
                continue
            if available >= required and available > 0:
                resource_fit *= required / available
            else:
                resource_fit *= 0.1

        estimated_cost = schedule_slot.get("estimated_cost", 500)
        cost_score = 1.0 - estimated_cost / self.max_bid_price
        quality_factor = schedule_slot.get("quality_factor", 0.8)

        return (
            self.objectives["schedule_efficiency"] * time_preference
            + self.objectives["resource_utilization"] * resource_fit
            + self.objectives["cost_minimization"] * cost_score
            + self.objectives["service_quality"] * quality_factor
        )

    def make_bid(self, auction_item, current_round):
        """Return a feasible bid, or None when the slot is unacceptable."""
        resource_requirements = auction_item["resource_requirements"]
        utility = self.calculate_utility(auction_item, resource_requirements)
        can_fulfill = all(
            self.resource_availability.get(resource_type, 0) >= required
            for resource_type, required in resource_requirements.items()
        )
        if not can_fulfill or utility < self.min_acceptable_utilization:
            return None

        base_price = auction_item.get("base_price", 500)
        if self.negotiation_strategy == "aggressive":
            bid_price = base_price * (1.0 + utility * 0.5)
        elif self.negotiation_strategy == "cooperative":
            bid_price = base_price * (0.8 + utility * 0.4)
        else:
            bid_price = base_price * (0.6 + utility * 0.3)
        bid_price = min(bid_price, self.max_bid_price)

        return {
            "agent_id": self.agent_id,
            "service_type": self.service_type,
            "bid_price": bid_price,
            "utility": utility,
            "strategy": self.negotiation_strategy,
            "resource_commitment": resource_requirements.copy(),
        }

    def update_strategy(self, negotiation_result):
        """Update reputation and strategy weights from a recorded outcome."""
        self.negotiation_history.append(negotiation_result)
        if negotiation_result.get("success", False):
            self.reputation_score = min(1.0, self.reputation_score + 0.05)
            self.success_history.append(1)
        else:
            self.reputation_score = max(0.0, self.reputation_score - 0.02)
            self.success_history.append(0)

        if len(self.success_history) >= 10:
            recent_success_rate = np.mean(list(self.success_history)[-10:])
            if (
                negotiation_result.get("strategy") == "aggressive"
                and recent_success_rate < 0.3
            ):
                self.strategy_weights["aggressive"] *= 0.9
                self.strategy_weights["cooperative"] *= 1.05
                self.strategy_weights["conservative"] *= 1.05
            total_weight = sum(self.strategy_weights.values())
            for strategy in self.strategy_weights:
                self.strategy_weights[strategy] /= total_weight
            self.negotiation_strategy = random.choices(
                list(self.strategy_weights),
                weights=list(self.strategy_weights.values()),
            )[0]

    def allocate_resources(self, resource_commitment):
        """Reserve the resources committed to an awarded slot."""
        for resource_type, amount in resource_commitment.items():
            if self.resource_availability.get(resource_type, 0) < amount:
                raise ValueError(f"Insufficient {resource_type} for {self.agent_id}.")
            self.resource_availability[resource_type] -= amount

    def release_resources(self, resource_commitment):
        """Return reserved resources after the service is completed."""
        for resource_type, amount in resource_commitment.items():
            self.resource_availability[resource_type] += amount
            if self.resource_availability[resource_type] > self.resources[resource_type]:
                raise ValueError(f"Resource balance exceeded for {resource_type}.")

    def get_agent_status(self):
        """Return a snapshot of the agent's current state."""
        return {
            "agent_id": self.agent_id,
            "service_type": self.service_type,
            "reputation_score": self.reputation_score,
            "current_strategy": self.negotiation_strategy,
            "resource_availability": self.resource_availability.copy(),
            "active_schedules": len(self.current_schedule),
            "success_rate": (
                float(np.mean(list(self.success_history)))
                if self.success_history
                else 0.0
            ),
        }

print("ServiceAgent class is ready.")

### 3. Sealed-bid auction manager

In a sealed-bid auction, participating agents submit bids without observing one another's bids. The rule used here selects the largest bid. For sorted bids \(b_{(1)} \geq b_{(2)} \geq \cdots\), the winning bid is \(b_{(1)}\), and the recorded reference price is \(b_{(2)}\) when at least two bids exist; with one bid, it is \(b_{(1)}\). This reference-price convention is not the payment rule in this simulation: the contract price is the winner's bid.

The manager also retains an English-auction method as part of the auction component. The ecosystem simulation below uses the sealed-bid method.

In [ ]:
class AuctionManager:
    """Run auctions and retain their results for later inspection."""

    def __init__(self, auction_type="sealed_bid"):
        self.auction_type = auction_type
        self.auction_history = deque(maxlen=100)
        self.market_clearing_price = None

    def run_sealed_bid_auction(self, auction_item, participating_agents):
        """Award the slot to the agent submitting the highest feasible bid."""
        bids = []
        for agent in participating_agents:
            bid = agent.make_bid(auction_item, 1)
            if bid is not None:
                bids.append(bid)
        if not bids:
            return {"success": False, "reason": "No valid bids received"}

        bids.sort(key=lambda item: item["bid_price"], reverse=True)
        winning_bid = bids[0]
        self.market_clearing_price = (
            bids[1]["bid_price"] if len(bids) > 1 else winning_bid["bid_price"]
        )
        auction_result = {
            "auction_type": "sealed_bid",
            "auction_item": auction_item,
            "winning_bid": winning_bid,
            "winner_agent_id": winning_bid["agent_id"],
            "market_clearing_price": self.market_clearing_price,
            "total_bids": len(bids),
            "success": True,
        }
        self.auction_history.append(auction_result)
        return auction_result

    def run_english_auction(self, auction_item, participating_agents, max_rounds=10):
        """Run an ascending-price auction with a fixed round limit."""
        current_price = auction_item.get("base_price", 100)
        price_increment = auction_item.get("price_increment", 50)
        active_agents = participating_agents.copy()
        auction_round = 0
        last_bids = []

        while len(active_agents) > 1 and auction_round < max_rounds:
            auction_round += 1
            round_bids = []
            for agent in active_agents:
                item = {**auction_item, "base_price": current_price}
                bid = agent.make_bid(item, auction_round)
                if bid is not None and bid["bid_price"] >= current_price:
                    round_bids.append(bid)
            if not round_bids:
                break
            last_bids = round_bids
            highest_bid = max(round_bids, key=lambda item: item["bid_price"])
            next_price = highest_bid["bid_price"] + price_increment
            active_ids = {
                bid["agent_id"] for bid in round_bids
                if bid["bid_price"] >= next_price
            }
            if not active_ids:
                break
            active_agents = [agent for agent in active_agents if agent.agent_id in active_ids]
            current_price = next_price

        if len(active_agents) == 1:
            winner = active_agents[0]
            final_bid = winner.make_bid(
                {**auction_item, "base_price": current_price}, auction_round
            )
            if final_bid is None:
                result = {"success": False, "reason": "No valid final bid"}
            else:
                result = {
                    "auction_type": "english",
                    "auction_item": auction_item,
                    "winning_bid": final_bid,
                    "winner_agent_id": winner.agent_id,
                    "final_price": current_price,
                    "rounds": auction_round,
                    "success": True,
                }
        else:
            result = {"success": False, "reason": "No winner determined"}
        self.auction_history.append(result)
        return result

print("AuctionManager class is ready.")

### 4. Simulated contract and ledger

A contract records the provider, requester, slot, resource commitment, price, and quality condition. The requester deposits the price plus a transaction fee; the price is held in escrow. On completion, the provider receives the full price if quality meets the requirement, or the price less the specified penalty otherwise. The penalty is credited to the system account, and the escrow is cleared. These bookkeeping rules make the simulated ledger internally auditable; they do not constitute a production smart contract.

In [ ]:
class BlockchainContractManager:
    """In-memory contract, escrow, and ledger simulation."""

    def __init__(self):
        self.contracts = {}
        self.contract_history = deque(maxlen=1000)
        self.ledger_balance = defaultdict(float)
        self.transaction_fee = 0.01
        self.total_transaction_fees = 0.0

    def create_service_contract(self, contract_details):
        """Create a contract and place its payment in escrow."""
        requester = contract_details["requester_agent_id"]
        price = float(contract_details["price"])
        fee = price * self.transaction_fee
        if self.ledger_balance[requester] < price + fee:
            raise ValueError("The requester does not have enough funds.")

        contract_id = f"CONTRACT_{len(self.contracts) + 1:06d}"
        contract = {
            "contract_id": contract_id,
            "service_provider": contract_details["provider_agent_id"],
            "service_requester": requester,
            "service_type": contract_details["service_type"],
            "schedule_slot": contract_details["schedule_slot"],
            "resource_commitment": contract_details["resource_commitment"].copy(),
            "price": price,
            "penalty_clause": contract_details.get("penalty_clause", 0.1),
            "quality_requirements": contract_details.get("quality_requirements", 0.8),
            "created_at": datetime.now(),
            "status": "active",
            "executed_at": None,
            "completion_verified": False,
        }
        self.ledger_balance[requester] -= price + fee
        self.ledger_balance[f"ESCROW_{contract_id}"] += price
        self.total_transaction_fees += fee
        self.contracts[contract_id] = contract
        self.contract_history.append({
            "action": "contract_created",
            "contract_id": contract_id,
            "timestamp": contract["created_at"],
        })
        return contract_id

    def execute_contract(self, contract_id, completion_quality=1.0):
        """Settle escrow once, applying the contract's quality condition."""
        if contract_id not in self.contracts:
            return {"success": False, "reason": "Contract not found"}
        contract = self.contracts[contract_id]
        if contract["status"] != "active":
            return {"success": False, "reason": "Contract not active"}

        quality_met = completion_quality >= contract["quality_requirements"]
        penalty = 0.0 if quality_met else contract["price"] * contract["penalty_clause"]
        payment = contract["price"] - penalty
        escrow_key = f"ESCROW_{contract_id}"
        if self.ledger_balance[escrow_key] < contract["price"]:
            raise ValueError("Escrow does not cover the contract price.")
        self.ledger_balance[contract["service_provider"]] += payment
        self.ledger_balance[contract["service_requester"]] += penalty
        self.ledger_balance[escrow_key] -= contract["price"]
        contract["status"] = (
            "completed_successfully" if quality_met else "completed_with_penalty"
        )
        contract["completion_verified"] = True
        contract["executed_at"] = datetime.now()
        contract["actual_quality"] = completion_quality
        self.contract_history.append({
            "action": "contract_executed",
            "contract_id": contract_id,
            "timestamp": contract["executed_at"],
        })
        return {
            "success": True,
            "payment": payment,
            "penalty_applied": penalty,
            "quality_met": quality_met,
        }

    def get_contract_status(self, contract_id):
        """Return key contract fields, or an error for an unknown identifier."""
        if contract_id not in self.contracts:
            return {"error": "Contract not found"}
        contract = self.contracts[contract_id]
        return {
            "contract_id": contract_id,
            "status": contract["status"],
            "provider": contract["service_provider"],
            "requester": contract["service_requester"],
            "price": contract["price"],
            "created_at": contract["created_at"],
            "executed_at": contract["executed_at"],
            "completion_verified": contract["completion_verified"],
        }

    def get_agent_balance(self, agent_id):
        """Return an agent's current ledger balance."""
        return self.ledger_balance.get(agent_id, 0.0)

    def get_market_statistics(self):
        """Summarize the contract population and realized completion times."""
        total = len(self.contracts)
        completed = [
            contract for contract in self.contracts.values()
            if contract["executed_at"] is not None
        ]
        average_hours = (
            float(np.mean([
                (contract["executed_at"] - contract["created_at"]).total_seconds() / 3600
                for contract in completed
            ]))
            if completed
            else 0.0
        )
        return {
            "total_contracts": total,
            "completed_contracts": len(completed),
            "completion_rate": len(completed) / total if total else 0.0,
            "average_completion_time_hours": average_hours,
            "total_transaction_fees": self.total_transaction_fees,
        }

print("BlockchainContractManager class is ready.")

### 5. Ecosystem initialization and simulation rules

The ecosystem coordinates agents through auctions rather than through a central optimizer. Slot requirements are generated in generic categories (`technicians`, `supervisors`, and `equipment`). Since providers use different resource names, `resource_mappings` translates each generic requirement into the corresponding provider resource. The requested quantity is preserved by this mapping; a bid is feasible only if the mapped resource is available.

At each hourly step, due contracts execute first, then eligible unallocated slots are auctioned. A successful award reserves its resources and creates an escrow-backed contract. The time series records reputation, utilization, completion, and coordination measures. These metrics describe the simulated rules and should not be interpreted as an optimality certificate.

In [ ]:
class AutonomousMultiAgentEcosystem:
    """Coordinate service agents, auctions, contracts, and simulation metrics."""

    def __init__(self):
        self.agents = {}
        self.auction_manager = AuctionManager()
        self.blockchain_manager = BlockchainContractManager()
        self.current_time = datetime.now().replace(minute=0, second=0, microsecond=0)
        self.schedule_slots = []
        self.active_contracts = {}
        self.system_metrics = deque(maxlen=1000)
        self.coordination_events = deque(maxlen=500)
        self.resource_mappings = {
            "MAINT_001": {"technicians": "technicians", "supervisors": "supervisors", "equipment": "tools"},
            "SECURITY_001": {"technicians": "guards", "supervisors": "vehicles", "equipment": "equipment"},
            "EMERGENCY_001": {"technicians": "paramedics", "supervisors": "vehicles", "equipment": "equipment"},
            "TUGBOAT_001": {"technicians": "crew", "supervisors": "vessels", "equipment": "fuel"},
            "POWER_001": {"technicians": "technicians", "supervisors": "equipment", "equipment": "fuel"},
            "ENVIRONMENT_001": {"technicians": "technicians", "supervisors": "vehicles", "equipment": "equipment"},
        }
        self.initialize_agents()
        self.initialize_schedule_slots()
        self.initialize_agent_balances()

    def initialize_agents(self):
        """Create the six service providers and their stated capabilities."""
        agent_configs = [
            {"agent_id": "MAINT_001", "service_type": "Maintenance", "capabilities": ["crane_maintenance", "equipment_repair", "inspection"], "resources": {"technicians": 8, "supervisors": 2, "tools": 15}},
            {"agent_id": "SECURITY_001", "service_type": "Security", "capabilities": ["patrol", "access_control", "surveillance"], "resources": {"guards": 12, "vehicles": 4, "equipment": 8}},
            {"agent_id": "EMERGENCY_001", "service_type": "Emergency", "capabilities": ["emergency_response", "fire_suppression", "medical_assistance"], "resources": {"paramedics": 6, "vehicles": 3, "equipment": 10}},
            {"agent_id": "TUGBOAT_001", "service_type": "Tugboat", "capabilities": ["vessel_positioning", "escort", "emergency_towing"], "resources": {"vessels": 3, "crew": 9, "fuel": 1000}},
            {"agent_id": "POWER_001", "service_type": "Power", "capabilities": ["backup_power", "grid_maintenance", "energy_management"], "resources": {"technicians": 5, "equipment": 12, "fuel": 500}},
            {"agent_id": "ENVIRONMENT_001", "service_type": "Environmental", "capabilities": ["pollution_monitoring", "waste_management", "compliance"], "resources": {"technicians": 4, "equipment": 8, "vehicles": 2}},
        ]
        for config in agent_configs:
            agent = ServiceAgent(
                config["agent_id"], config["service_type"],
                config["capabilities"], config["resources"]
            )
            self.agents[config["agent_id"]] = agent

    def initialize_schedule_slots(self):
        """Create one hourly slot of each service priority for 24 hours."""
        for hour in range(24):
            for slot_type in ["routine", "priority", "emergency"]:
                slot = {
                    "slot_id": f"SLOT_{hour:02d}_{slot_type.upper()}",
                    "time": hour,
                    "type": slot_type,
                    "base_price": 200 if slot_type == "routine" else 500 if slot_type == "priority" else 1000,
                    "resource_requirements": self.generate_resource_requirements(slot_type),
                    "quality_factor": 0.8 if slot_type == "routine" else 0.9 if slot_type == "priority" else 0.95,
                    "duration": 1,
                    "allocated": False,
                }
                self.schedule_slots.append(slot)

    def generate_resource_requirements(self, slot_type):
        """Draw generic resource quantities for the requested slot category."""
        if slot_type == "routine":
            return {"technicians": random.randint(2, 4), "supervisors": random.randint(0, 1), "equipment": random.randint(3, 6)}
        if slot_type == "priority":
            return {"technicians": random.randint(4, 7), "supervisors": random.randint(1, 2), "equipment": random.randint(6, 10)}
        return {"technicians": random.randint(6, 10), "supervisors": random.randint(2, 3), "equipment": random.randint(8, 15)}

    def initialize_agent_balances(self):
        """Fund the system requester so it can place contract payments in escrow."""
        self.blockchain_manager.ledger_balance["SYSTEM"] = 1_000_000.0
        for agent_id in self.agents:
            self.blockchain_manager.ledger_balance[agent_id] = 10_000.0

    def map_resource_requirements(self, agent, generic_requirements):
        """Translate generic slot categories into this provider's resource names."""
        mapping = self.resource_mappings[agent.agent_id]
        return {
            mapping[category]: amount
            for category, amount in generic_requirements.items()
            if amount > 0
        }

    def run_auction_round(self, slot):
        """Auction one slot among the provider types allowed by its priority."""
        if slot["allocated"]:
            return None
        if slot["type"] == "emergency":
            eligible_types = ["Emergency", "Tugboat"]
        elif slot["type"] == "priority":
            eligible_types = ["Emergency", "Tugboat", "Security", "Power"]
        else:
            eligible_types = [agent.service_type for agent in self.agents.values()]

        participating_agents = [
            agent for agent in self.agents.values()
            if agent.service_type in eligible_types
        ]
        bids_by_agent = {}
        for agent in participating_agents:
            item = slot.copy()
            item["resource_requirements"] = self.map_resource_requirements(
                agent, slot["resource_requirements"]
            )
            bids_by_agent[agent.agent_id] = item

        # Each provider receives its own service-specific resource commitment.
        bids = []
        for agent in participating_agents:
            bid = agent.make_bid(bids_by_agent[agent.agent_id], 1)
            if bid is not None:
                bids.append(bid)
        if not bids:
            self.auction_manager.auction_history.append({"success": False, "reason": "No valid bids received", "auction_item": slot})
            return {"success": False, "reason": "No valid bids received"}

        bids.sort(key=lambda item: item["bid_price"], reverse=True)
        winning_bid = bids[0]
        self.auction_manager.market_clearing_price = (
            bids[1]["bid_price"] if len(bids) > 1 else winning_bid["bid_price"]
        )
        auction_result = {
            "auction_type": "sealed_bid", "auction_item": slot,
            "winning_bid": winning_bid, "winner_agent_id": winning_bid["agent_id"],
            "market_clearing_price": self.auction_manager.market_clearing_price,
            "total_bids": len(bids), "success": True,
        }
        self.auction_manager.auction_history.append(auction_result)

        contract_details = {
            "provider_agent_id": winning_bid["agent_id"],
            "requester_agent_id": "SYSTEM",
            "service_type": slot["type"],
            "schedule_slot": slot,
            "resource_commitment": winning_bid["resource_commitment"],
            "price": winning_bid["bid_price"],
            "penalty_clause": 0.1,
            "quality_requirements": slot["quality_factor"],
        }
        contract_id = self.blockchain_manager.create_service_contract(contract_details)
        winner_agent = self.agents[winning_bid["agent_id"]]
        winner_agent.allocate_resources(winning_bid["resource_commitment"])
        slot["allocated"] = True
        slot["allocated_to"] = winning_bid["agent_id"]
        slot["contract_id"] = contract_id
        self.active_contracts[contract_id] = {
            "contract_id": contract_id,
            "agent_id": winning_bid["agent_id"],
            "slot": slot,
            "resource_commitment": winning_bid["resource_commitment"].copy(),
            "execution_time": self.current_time + timedelta(hours=1),
            "status": "pending",
        }
        self.coordination_events.append({
            "event_type": "auction_won", "agent_id": winning_bid["agent_id"],
            "slot_id": slot["slot_id"], "price": winning_bid["bid_price"],
            "timestamp": self.current_time,
        })
        winner_agent.update_strategy({
            "success": True, "strategy": winning_bid["strategy"],
            "price": winning_bid["bid_price"],
        })
        return {
            "success": True, "contract_id": contract_id,
            "winner": winning_bid["agent_id"], "price": winning_bid["bid_price"],
        }

    def execute_pending_contracts(self):
        """Execute due contracts, settle payments, and release reserved resources."""
        executed_contracts = []
        for contract_id, contract_info in list(self.active_contracts.items()):
            if contract_info["status"] != "pending" or contract_info["execution_time"] > self.current_time:
                continue
            agent = self.agents[contract_info["agent_id"]]
            execution_quality = random.uniform(0.7, 1.0)
            result = self.blockchain_manager.execute_contract(contract_id, execution_quality)
            if not result["success"]:
                continue
            agent.release_resources(contract_info["resource_commitment"])
            agent.update_strategy({
                "success": result["quality_met"],
                "quality_met": result["quality_met"],
                "payment": result["payment"],
            })
            contract_info["status"] = "executed"
            executed_contracts.append({
                "contract_id": contract_id,
                "agent_id": contract_info["agent_id"],
                "execution_quality": execution_quality,
                "payment": result["payment"],
                "quality_met": result["quality_met"],
                "penalty_applied": result["penalty_applied"],
            })
            self.coordination_events.append({
                "event_type": "contract_executed", "contract_id": contract_id,
                "agent_id": contract_info["agent_id"], "quality": execution_quality,
                "timestamp": self.current_time,
            })
        return executed_contracts

    def run_simulation_step(self):
        """Advance one hour, settle due contracts, and auction eligible slots."""
        self.current_time += timedelta(hours=1)
        executed_contracts = self.execute_pending_contracts()
        auction_results = []
        elapsed_hour = int((self.current_time - self.simulation_start).total_seconds() // 3600)
        for slot in self.schedule_slots:
            if not slot["allocated"] and slot["time"] < elapsed_hour:
                result = self.run_auction_round(slot)
                if result is not None:
                    auction_results.append(result)
        metrics = self.calculate_system_metrics()
        self.system_metrics.append({"timestamp": self.current_time, "metrics": metrics})
        return {
            "timestamp": self.current_time,
            "executed_contracts": executed_contracts,
            "auction_results": auction_results,
            "system_metrics": metrics,
        }

    def calculate_system_metrics(self):
        """Calculate aggregate reputation, resource use, and market measures."""
        reputations = [agent.reputation_score for agent in self.agents.values()]
        total_resources = defaultdict(int)
        used_resources = defaultdict(int)
        for agent in self.agents.values():
            for resource_type, total in agent.resources.items():
                total_resources[resource_type] += total
                used_resources[resource_type] += total - agent.resource_availability[resource_type]
        utilization = {
            resource_type: used_resources[resource_type] / total
            for resource_type, total in total_resources.items()
            if total > 0
        }
        average_utilization = float(np.mean(list(utilization.values()))) if utilization else 0.0
        market = self.blockchain_manager.get_market_statistics()
        recent_events = [
            event for event in self.coordination_events
            if 0 <= (self.current_time - event["timestamp"]).total_seconds() < 3600
        ]
        return {
            "avg_agent_reputation": float(np.mean(reputations)) if reputations else 0.0,
            "avg_resource_utilization": average_utilization,
            "resource_utilization_by_type": utilization,
            "contract_completion_rate": market["completion_rate"],
            "total_contracts": market["total_contracts"],
            "coordination_events_per_hour": len(recent_events),
            "active_agents": sum(
                agent.resource_availability != agent.resources
                for agent in self.agents.values()
            ),
            "market_efficiency": market["completion_rate"] * average_utilization,
        }

    def get_ecosystem_status(self):
        """Return a compact snapshot of agents, slots, and market activity."""
        return {
            "current_time": self.current_time,
            "total_agents": len(self.agents),
            "agent_statuses": {
                agent_id: agent.get_agent_status()
                for agent_id, agent in self.agents.items()
            },
            "active_contracts": sum(
                contract["status"] == "pending"
                for contract in self.active_contracts.values()
            ),
            "total_slots": len(self.schedule_slots),
            "allocated_slots": sum(slot["allocated"] for slot in self.schedule_slots),
            "market_stats": self.blockchain_manager.get_market_statistics(),
            "recent_metrics": self.system_metrics[-1]["metrics"] if self.system_metrics else None,
        }

        # The elapsed-hour schedule uses this fixed simulation origin.

    def _set_simulation_origin(self):
        self.simulation_start = self.current_time

AutonomousMultiAgentEcosystem._set_simulation_origin = _set_simulation_origin
original_init = AutonomousMultiAgentEcosystem.__init__

def _initialize_with_origin(self):
    original_init(self)
    self.simulation_start = self.current_time

AutonomousMultiAgentEcosystem.__init__ = _initialize_with_origin

print("AutonomousMultiAgentEcosystem class is ready.")

### 6. Initialize the ecosystem

The initial status shows the agents, their capabilities, and their available resources. The system account funds contract escrow; provider accounts are initialized separately. The slot count follows from the construction rule: 24 hours multiplied by three priority classes.

In [ ]:
ecosystem = AutonomousMultiAgentEcosystem()

print("Autonomous multi-agent ecosystem initialized.")
print(f"Total agents: {len(ecosystem.agents)}")
print(f"Schedule slots: {len(ecosystem.schedule_slots)}")
print(f"System requester balance: ${ecosystem.blockchain_manager.get_agent_balance('SYSTEM'):,.2f}")

print("\nAgent information")
print("-" * 60)
for agent_id, agent in ecosystem.agents.items():
    status = agent.get_agent_status()
    print(f"\n{agent_id} ({status['service_type']}):")
    print(f"  Reputation: {status['reputation_score']:.3f}")
    print(f"  Strategy: {status['current_strategy']}")
    print(f"  Resources: {status['resource_availability']}")
    print(f"  Capabilities: {agent.capabilities}")

### 7. Run a 24-hour simulation

Each call to `run_simulation_step` advances the clock by one hour. Pending contracts are executed before new slot auctions, so resources are held until their service has completed. The simulation records successful awards and executed contracts separately: a slot can be awarded without its contract having executed by the end of the observation period.

In [ ]:
simulation_steps = 24
simulation_results = []
total_auctions = 0
total_contracts = 0

print("=" * 80)
print("24-HOUR AUTONOMOUS MULTI-AGENT ECOSYSTEM SIMULATION")
print("=" * 80)
print(f"Running {simulation_steps} hourly steps.\n")

for step in range(simulation_steps):
    result = ecosystem.run_simulation_step()
    simulation_results.append(result)
    total_auctions += sum(
        item.get("success", False) for item in result["auction_results"]
    )
    total_contracts += len(result["executed_contracts"])

    if (step + 1) % 6 == 0:
        metrics = result["system_metrics"]
        print(f"Hour {step + 1:2d}:")
        print(f"  Average agent reputation: {metrics['avg_agent_reputation']:.3f}")
        print(f"  Average resource utilization: {metrics['avg_resource_utilization']:.3f}")
        print(f"  Contract completion rate: {metrics['contract_completion_rate']:.3f}")
        print(f"  Market efficiency: {metrics['market_efficiency']:.3f}")
        print(f"  Coordination events in the last hour: {metrics['coordination_events_per_hour']}\n")

print("Simulation completed.")
print(f"Successful slot awards: {total_auctions}")
print(f"Contracts executed during the simulation: {total_contracts}")
print(f"Simulation steps: {len(simulation_results)}")

assert len(simulation_results) == simulation_steps
assert all(
    0.0 <= agent.reputation_score <= 1.0
    for agent in ecosystem.agents.values()
)
assert all(
    0 <= agent.resource_availability[key] <= amount
    for agent in ecosystem.agents.values()
    for key, amount in agent.resources.items()
)
print("State checks passed: reputation and resource balances remain within their bounds.")

### 8. Performance analysis

The analysis summarizes the recorded hourly metrics and final agent and contract states. The mean and standard deviation describe variation across this run's hourly observations; they are not estimates across independent replications. Reputation is bounded by the update rule. Resource utilization is the fraction of each resource category currently reserved, averaged across categories, so it can fall when contracts finish and resources are released.

In [ ]:
def analyze_ecosystem_performance(simulation_results, ecosystem):
    """Print and return descriptive statistics for one simulation run."""
    if not simulation_results:
        raise ValueError("At least one simulation result is required.")

    metrics_series = [result["system_metrics"] for result in simulation_results]
    metric_names = {
        "agent_reputation": "avg_agent_reputation",
        "resource_utilization": "avg_resource_utilization",
        "market_efficiency": "market_efficiency",
        "contract_completion_rate": "contract_completion_rate",
    }
    statistics = {}
    for label, key in metric_names.items():
        values = np.array([metrics[key] for metrics in metrics_series], dtype=float)
        statistics[label] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
        }

    market_stats = ecosystem.blockchain_manager.get_market_statistics()
    auction_events = [
        event for event in ecosystem.coordination_events
        if event["event_type"] == "auction_won"
    ]
    execution_events = [
        event for event in ecosystem.coordination_events
        if event["event_type"] == "contract_executed"
    ]

    print("=" * 72)
    print("AUTONOMOUS MULTI-AGENT ECOSYSTEM PERFORMANCE ANALYSIS")
    print("=" * 72)
    for label, values in statistics.items():
        print(f"{label.replace('_', ' ').title()}: mean={values['mean']:.3f}, std={values['std']:.3f}")

    print("\nAgent performance")
    for agent_id, agent in ecosystem.agents.items():
        status = agent.get_agent_status()
        print(
            f"{agent_id} ({status['service_type']}): "
            f"reputation={status['reputation_score']:.3f}, "
            f"success rate={status['success_rate']:.3f}, "
            f"strategy={status['current_strategy']}"
        )

    print("\nContract and coordination statistics")
    print(f"Total contracts: {market_stats['total_contracts']}")
    print(f"Completed contracts: {market_stats['completed_contracts']}")
    print(f"Completion rate: {market_stats['completion_rate']:.3f}")
    print(f"Average completion time: {market_stats['average_completion_time_hours']:.4f} hours")
    print(f"Transaction fees: ${market_stats['total_transaction_fees']:.2f}")
    print(f"Successful auction events: {len(auction_events)}")
    print(f"Contract execution events: {len(execution_events)}")
    if auction_events:
        print(f"Mean winning bid: ${np.mean([event['price'] for event in auction_events]):.2f}")
    if execution_events:
        print(f"Mean execution quality: {np.mean([event['quality'] for event in execution_events]):.3f}")

    statistics["total_contracts"] = market_stats["total_contracts"]
    statistics["completion_rate"] = market_stats["completion_rate"]
    statistics["total_auctions"] = len(auction_events)
    statistics["total_executions"] = len(execution_events)
    return statistics


performance_stats = analyze_ecosystem_performance(simulation_results, ecosystem)

### 9. Performance dashboard

The dashboard places time-series indicators and end-of-run summaries side by side. The two event types in the final panel use different units—auction price is monetary, while execution quality is a unitless score—so that panel is descriptive rather than a direct scale comparison. Bar colors in the agent panel identify the current strategy and are explained by the legend.

In [ ]:
metrics_series = [result["system_metrics"] for result in simulation_results]
timestamps = [result["timestamp"] for result in simulation_results]
hours = [
    (timestamp - timestamps[0]).total_seconds() / 3600
    for timestamp in timestamps
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Autonomous Multi-Agent Ecosystem: Performance Dashboard", fontsize=16, fontweight="bold")

ax = axes[0, 0]
reputations = [item["avg_agent_reputation"] for item in metrics_series]
ax.plot(hours, reputations, color="tab:blue", linewidth=2, marker="o", markersize=4)
ax.fill_between(hours, reputations, alpha=0.2, color="tab:blue")
ax.set(title="Agent Reputation", xlabel="Elapsed time (hours)", ylabel="Mean reputation score")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
resource_utils = [item["avg_resource_utilization"] for item in metrics_series]
ax.plot(hours, resource_utils, color="tab:green", linewidth=2, marker="s", markersize=4)
ax.fill_between(hours, resource_utils, alpha=0.2, color="tab:green")
ax.set(title="Resource Utilization", xlabel="Elapsed time (hours)", ylabel="Mean reserved fraction")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

ax = axes[0, 2]
market_efficiency = [item["market_efficiency"] for item in metrics_series]
completion_rates = [item["contract_completion_rate"] for item in metrics_series]
ax.plot(hours, market_efficiency, color="tab:red", marker="^", label="Market efficiency")
ax.plot(hours, completion_rates, color="tab:orange", marker="d", label="Contract completion rate")
ax.set(title="Market Measures", xlabel="Elapsed time (hours)", ylabel="Score or rate")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
agent_names = list(ecosystem.agents)
agent_reputations = [ecosystem.agents[name].reputation_score for name in agent_names]
agent_success_rates = [ecosystem.agents[name].get_agent_status()["success_rate"] for name in agent_names]
strategies = [ecosystem.agents[name].negotiation_strategy for name in agent_names]
strategy_colors = {"aggressive": "tab:red", "cooperative": "tab:green", "conservative": "tab:blue"}
bars = ax.bar(
    agent_names, agent_reputations,
    color=[strategy_colors[strategy] for strategy in strategies],
    alpha=0.75, edgecolor="black",
)
ax.set(title="Agent Reputation by Current Strategy", ylabel="Reputation score")
ax.set_ylim(0, 1.12)
ax.tick_params(axis="x", rotation=45)
ax.grid(True, axis="y", alpha=0.3)
for bar, success_rate in zip(bars, agent_success_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"Success {success_rate:.2f}", ha="center", va="bottom", fontsize=7)
strategy_handles = [
    plt.Rectangle((0, 0), 1, 1, color=color, label=strategy.title())
    for strategy, color in strategy_colors.items()
]
ax.legend(handles=strategy_handles, title="Current strategy", fontsize=8)

ax = axes[1, 1]
latest_utilization = metrics_series[-1]["resource_utilization_by_type"]
if latest_utilization:
    resource_types = list(latest_utilization)
    values = [latest_utilization[key] for key in resource_types]
    bars = ax.bar(resource_types, values, color="tab:purple", alpha=0.75, edgecolor="black")
    ax.set_ylim(0, 1.12)
    ax.tick_params(axis="x", rotation=45)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{value:.3f}", ha="center", va="bottom", fontsize=8)
else:
    ax.text(0.5, 0.5, "No resource categories", ha="center", va="center", transform=ax.transAxes)
ax.set(title="Resource Utilization by Type", ylabel="Reserved fraction")
ax.grid(True, axis="y", alpha=0.3)

ax = axes[1, 2]
auction_events = [event for event in ecosystem.coordination_events if event["event_type"] == "auction_won"]
execution_events = [event for event in ecosystem.coordination_events if event["event_type"] == "contract_executed"]
if auction_events:
    event_hours = [(event["timestamp"] - timestamps[0]).total_seconds() / 3600 for event in auction_events]
    ax.scatter(event_hours, [event["price"] for event in auction_events],
               color="tab:blue", label="Winning bid (currency units)", marker="o", alpha=0.7)
if execution_events:
    event_hours = [(event["timestamp"] - timestamps[0]).total_seconds() / 3600 for event in execution_events]
    ax.scatter(event_hours, [event["quality"] for event in execution_events],
               color="tab:red", label="Execution quality (unitless)", marker="s", alpha=0.7)
ax.set(title="Coordination Events", xlabel="Elapsed time (hours)", ylabel="Value (units differ)")
if auction_events or execution_events:
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, "No coordination events", ha="center", va="center", transform=ax.transAxes)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Summary and interpretation

### What the simulation demonstrates

- **Local decision-making:** each provider evaluates a slot using its own utility weights, strategy, and available resources.
- **Auction-based allocation:** the sealed-bid rule selects the highest feasible bid among the eligible agents. The winner's bid determines the simulated contract price; the recorded second-highest bid is a reference statistic.
- **Resource reservation:** an awarded commitment reduces the provider's available resources until the contract is executed.
- **Quality-linked settlement:** execution quality determines whether the provider receives the full price or a reduced payment under the penalty clause.
- **Reputation and adaptation:** outcomes update reputation, and the strategy-weight update rule can change an agent's strategy after a sufficiently long history.

### Interpreting the reported measures

`avg_agent_reputation` is the mean reputation of the six agents. `avg_resource_utilization` is the mean of the reserved fractions across resource categories at that simulation step. `contract_completion_rate` is completed contracts divided by all contracts created so far. `market_efficiency` is defined in the code as the product of completion rate and average resource utilization; it is a descriptive index, not an optimization objective or a standard efficiency measure. The displayed averages and standard deviations summarize hourly observations from one run, not repeated trials.

Because slot requirements and execution quality are stochastic, results can vary between runs. A strategy that is absent from the final agent snapshot may still have been used earlier; the final snapshot reports only each agent's current strategy. The model's fixed rules also mean that its outcomes depend on the chosen eligibility lists, mappings, utility weights, and bid formulas.

### Scope and limitations

This is a teaching simulation, not an operational port scheduling system. It does not establish that decentralized decisions are globally optimal, that the auction is incentive-compatible, or that the in-memory ledger has blockchain security. It omits interactions among simultaneous services, detailed temporal conflicts, travel and setup times, budget constraints beyond the system balance, and statistical uncertainty across independent replications. Those assumptions should be revisited before using this model to support real operational decisions.

### Further study

Natural extensions within this agent-based framework include adding service dependencies, more detailed resource calendars, and repeated independent simulation runs to study variability. Such extensions would change the modeled environment and require their assumptions and evaluation measures to be stated explicitly.